In [1]:
import json

with open('../data/GraphDTA/data/davis/proteins.txt') as f:
    proteins = json.load(f)

print('Total number of proteins in Davis:', len(proteins))

first_protein_name = list(proteins.keys())[0]
first_sequence = proteins[first_protein_name]

print('Protein name:', first_protein_name)
print('Sequence length:', len(first_sequence))
print('First 60 characters:', first_sequence[:60])

Total number of proteins in Davis: 442
Protein name: AAK1
Sequence length: 961
First 60 characters: MKKFFDSRREQGGSGLGSGSSGGGGSTSGLGSGYIGRVFGIGRQQVTVDEVLAEGGFAIV


In [2]:
all_letters = set()
for seq in proteins.values():
    all_letters.update(seq)

print('Number of unique letters found:', len(all_letters))
print('Letters:', sorted(all_letters))

Number of unique letters found: 21
Letters: ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'X', 'Y']


In [3]:
x_count = 0
total_count = 0
proteins_with_x = 0

for name, seq in proteins.items():
    total_count += len(seq)
    if 'X' in seq:
        proteins_with_x += 1
        x_count += seq.count('X')

print('Total amino acid letters across all proteins:', total_count)
print('Total X occurrences:', x_count)
print('Number of proteins containing at least one X:', proteins_with_x, 'out of', len(proteins))
print(f'X makes up {100*x_count/total_count:.4f}% of all letters')

Total amino acid letters across all proteins: 348715
Total X occurrences: 2
Number of proteins containing at least one X: 1 out of 442
X makes up 0.0006% of all letters


In [4]:
AMINO_ACID_VOCAB = sorted(all_letters)
print('Vocabulary:', AMINO_ACID_VOCAB)
print('Vocabulary size:', len(AMINO_ACID_VOCAB))

aa_to_index = {aa: i for i, aa in enumerate(AMINO_ACID_VOCAB)}
print()
print('Lookup table (letter -> index number):')
print(aa_to_index)

Vocabulary: ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'X', 'Y']
Vocabulary size: 21

Lookup table (letter -> index number):
{'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 'M': 10, 'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'X': 19, 'Y': 20}


In [5]:
sequence_as_indices = [aa_to_index[letter] for letter in first_sequence]

print('Original sequence (first 20 letters):', first_sequence[:20])
print('As indices (first 20):', sequence_as_indices[:20])
print()
print('Total length check:', len(sequence_as_indices), '==', len(first_sequence))

Original sequence (first 20 letters): MKKFFDSRREQGGSGLGSGS
As indices (first 20): [10, 8, 8, 4, 4, 2, 15, 14, 14, 3, 13, 5, 5, 15, 5, 9, 5, 15, 5, 15]

Total length check: 961 == 961


In [6]:
import torch
import torch.nn as nn

embedding_dim = 8
embedding_layer = nn.Embedding(num_embeddings=21, embedding_dim=embedding_dim)

sequence_tensor = torch.tensor(sequence_as_indices, dtype=torch.long)
print('sequence_tensor shape:', sequence_tensor.shape)

embedded_sequence = embedding_layer(sequence_tensor)
print('embedded_sequence shape:', embedded_sequence.shape)
print()
print('Vector for first letter (M):', embedded_sequence[0].tolist())

sequence_tensor shape: torch.Size([961])
embedded_sequence shape: torch.Size([961, 8])

Vector for first letter (M): [0.2668028771877289, -0.39173150062561035, 0.8446629047393799, -2.2308197021484375, 0.13591890037059784, 0.16837109625339508, -0.4183732867240906, -1.5986522436141968]


In [7]:
import torch.nn.functional as F

# Conv1d expects shape: (batch, channels, length) -- so we reshape and transpose
x = embedded_sequence.unsqueeze(0).transpose(1, 2)
print('Shape before conv:', x.shape)

conv_layer = nn.Conv1d(in_channels=8, out_channels=16, kernel_size=8)

conv_output = conv_layer(x)
print('Shape after conv:', conv_output.shape)

Shape before conv: torch.Size([1, 8, 961])
Shape after conv: torch.Size([1, 16, 954])


In [8]:
max_pooled = conv_output.max(dim=2).values
print('Shape after max pooling:', max_pooled.shape)
print()
print('Protein vector (max pooled):', max_pooled.squeeze().tolist())

Shape after max pooling: torch.Size([1, 16])

Protein vector (max pooled): [1.4811736345291138, 1.600411295890808, 1.9477134943008423, 1.665602207183838, 1.4423664808273315, 1.5353775024414062, 1.294004201889038, 1.403808355331421, 1.7649105787277222, 1.7512423992156982, 2.0551533699035645, 1.6458946466445923, 1.6028459072113037, 2.000789165496826, 1.2214100360870361, 1.1922259330749512]


In [9]:
# conv_output is [1, 16, 954] -- rearrange to [954, 16] for easier pooling, same shape style as our atom vectors
positions = conv_output.squeeze(0).transpose(0, 1)
print('Positions shape:', positions.shape)

attention_scorer_protein = nn.Linear(16, 1)
raw_scores = attention_scorer_protein(positions)
attention_weights = F.softmax(raw_scores, dim=0)

print('Sum of weights:', attention_weights.sum().item())

attention_pooled_protein = (attention_weights * positions).sum(dim=0)
print('Protein vector (attention pooled):', attention_pooled_protein.tolist())

Positions shape: torch.Size([954, 16])
Sum of weights: 1.0000003576278687
Protein vector (attention pooled): [-0.020772237330675125, -0.0838276743888855, 0.21947114169597626, 0.122604139149189, 0.12195280939340591, 0.017851730808615685, 0.03585295379161835, -0.30372557044029236, 0.19997166097164154, 0.009920813143253326, 0.16105486452579498, -0.14466531574726105, -0.12090032547712326, 0.10942872613668442, 0.021822791546583176, 0.05789174884557724]


In [10]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

with open('../data/GraphDTA/data/davis/ligands_can.txt') as f:
    ligands = json.load(f)

drug_smiles = ligands[list(ligands.keys())[0]]
mol = Chem.MolFromSmiles(drug_smiles)

ATOM_VOCAB = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'P', 'I']

def one_hot(value, vocab):
    vec = [0] * (len(vocab) + 1)
    if value in vocab:
        vec[vocab.index(value)] = 1
    else:
        vec[-1] = 1
    return vec

def atom_features(atom):
    features = []
    features += one_hot(atom.GetSymbol(), ATOM_VOCAB)
    features += [atom.GetDegree()]
    features += [atom.GetFormalCharge()]
    features += [int(atom.GetIsAromatic())]
    features += [int(atom.IsInRing())]
    return features

all_atom_features = [atom_features(atom) for atom in mol.GetAtoms()]

edge_sources, edge_targets = [], []
for bond in mol.GetBonds():
    i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
    edge_sources += [i, j]
    edge_targets += [j, i]

drug_x = torch.tensor(all_atom_features, dtype=torch.float)
drug_edge_index = torch.tensor([edge_sources, edge_targets], dtype=torch.long)

drug_conv1 = GCNConv(14, 16)
drug_conv2 = GCNConv(16, 16)
drug_conv3 = GCNConv(16, 16)

h = F.relu(drug_conv1(drug_x, drug_edge_index))
h = F.relu(drug_conv2(h, drug_edge_index))
h = drug_conv3(h, drug_edge_index)

drug_attention_scorer = nn.Linear(16, 1)
drug_scores = F.softmax(drug_attention_scorer(h), dim=0)
drug_vector = (drug_scores * h).sum(dim=0)

print('Drug vector shape:', drug_vector.shape)
print('Drug vector:', drug_vector.tolist())

Drug vector shape: torch.Size([16])
Drug vector: [-0.18102233111858368, 0.33528459072113037, 0.11459775269031525, -0.2474783957004547, -0.08774711191654205, 0.061168938875198364, 0.10857295989990234, -0.2527430057525635, -0.27211475372314453, -0.13646456599235535, -0.32299596071243286, 0.24554771184921265, 0.12190309166908264, 0.09510837495326996, 0.24427449703216553, -0.43115299940109253]


In [11]:
# Step 1: concatenate the two vectors into one
combined_vector = torch.cat([drug_vector, attention_pooled_protein])
print('Drug vector length:', drug_vector.shape)
print('Protein vector length:', attention_pooled_protein.shape)
print('Combined vector length:', combined_vector.shape)
print()
print('Combined vector:', combined_vector.tolist())

Drug vector length: torch.Size([16])
Protein vector length: torch.Size([16])
Combined vector length: torch.Size([32])

Combined vector: [-0.18102233111858368, 0.33528459072113037, 0.11459775269031525, -0.2474783957004547, -0.08774711191654205, 0.061168938875198364, 0.10857295989990234, -0.2527430057525635, -0.27211475372314453, -0.13646456599235535, -0.32299596071243286, 0.24554771184921265, 0.12190309166908264, 0.09510837495326996, 0.24427449703216553, -0.43115299940109253, -0.020772237330675125, -0.0838276743888855, 0.21947114169597626, 0.122604139149189, 0.12195280939340591, 0.017851730808615685, 0.03585295379161835, -0.30372557044029236, 0.19997166097164154, 0.009920813143253326, 0.16105486452579498, -0.14466531574726105, -0.12090032547712326, 0.10942872613668442, 0.021822791546583176, 0.05789174884557724]


In [12]:
# Step 2: MLP - shrink the combined vector down to a single predicted affinity number
mlp = nn.Sequential(
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

predicted_pKd = mlp(combined_vector)
print('Predicted pKd:', predicted_pKd.item())

Predicted pKd: -0.1163695678114891


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class DrugEncoder(nn.Module):
    def __init__(self, atom_feature_dim=14, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(atom_feature_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.attention_scorer = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        h = self.conv3(h, edge_index)

        scores = F.softmax(self.attention_scorer(h), dim=0)
        drug_vector = (scores * h).sum(dim=0)
        return drug_vector


class ProteinEncoder(nn.Module):
    def __init__(self, vocab_size=21, embed_dim=8, hidden_dim=16, kernel_size=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv = nn.Conv1d(embed_dim, hidden_dim, kernel_size=kernel_size)

    def forward(self, sequence_indices):
        # sequence_indices: shape [seq_len]
        embedded = self.embedding(sequence_indices)            # [seq_len, embed_dim]
        x = embedded.unsqueeze(0).transpose(1, 2)               # [1, embed_dim, seq_len]
        conv_out = self.conv(x)                                  # [1, hidden_dim, num_positions]
        pooled = conv_out.max(dim=2).values                      # [1, hidden_dim]
        return pooled.squeeze(0)                                 # [hidden_dim]


class DTIModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.drug_encoder = DrugEncoder()
        self.protein_encoder = ProteinEncoder()
        self.mlp = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, drug_x, drug_edge_index, protein_sequence_indices):
        drug_vec = self.drug_encoder(drug_x, drug_edge_index)
        protein_vec = self.protein_encoder(protein_sequence_indices)
        combined = torch.cat([drug_vec, protein_vec])
        predicted_pKd = self.mlp(combined)
        return predicted_pKd

In [14]:
model = DTIModel()

predicted = model(drug_x, drug_edge_index, sequence_tensor)
print('Predicted pKd:', predicted.item())

Predicted pKd: -0.15330377221107483


In [15]:
import json
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem
from torch_geometric.nn import GCNConv

DATA_DIR = '../data/GraphDTA/data/davis'

with open(f'{DATA_DIR}/ligands_can.txt') as f:
    ligands = json.load(f)

with open(f'{DATA_DIR}/proteins.txt') as f:
    proteins = json.load(f)

with open(f'{DATA_DIR}/Y', 'rb') as f:
    Y = pickle.load(f, encoding='latin1')

with open(f'{DATA_DIR}/folds/train_fold_setting1.txt') as f:
    train_folds = json.load(f)

with open(f'{DATA_DIR}/folds/test_fold_setting1.txt') as f:
    test_fold = json.load(f)

# flatten train_folds (it's 5 separate folds) into one big list of training indices
train_indices = [idx for fold in train_folds for idx in fold]

print('Num drugs:', len(ligands))
print('Num proteins:', len(proteins))
print('Y shape:', Y.shape)
print('Num training indices:', len(train_indices))
print('Num test indices:', len(test_fold))

Num drugs: 68
Num proteins: 442
Y shape: (68, 442)
Num training indices: 25046
Num test indices: 5010


In [16]:
ATOM_VOCAB = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'P', 'I']
AMINO_ACID_VOCAB = sorted(set(letter for seq in proteins.values() for letter in seq))
aa_to_index = {aa: i for i, aa in enumerate(AMINO_ACID_VOCAB)}

ligand_ids = list(ligands.keys())
protein_names = list(proteins.keys())

def one_hot(value, vocab):
    vec = [0] * (len(vocab) + 1)
    if value in vocab:
        vec[vocab.index(value)] = 1
    else:
        vec[-1] = 1
    return vec

def atom_features(atom):
    features = []
    features += one_hot(atom.GetSymbol(), ATOM_VOCAB)
    features += [atom.GetDegree()]
    features += [atom.GetFormalCharge()]
    features += [int(atom.GetIsAromatic())]
    features += [int(atom.IsInRing())]
    return features

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    all_atom_features = [atom_features(atom) for atom in mol.GetAtoms()]
    edge_sources, edge_targets = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_sources += [i, j]
        edge_targets += [j, i]
    x = torch.tensor(all_atom_features, dtype=torch.float)
    edge_index = torch.tensor([edge_sources, edge_targets], dtype=torch.long)
    return x, edge_index

def sequence_to_indices(sequence):
    return torch.tensor([aa_to_index[letter] for letter in sequence], dtype=torch.long)

def get_example(flat_index):
    drug_row, protein_col = np.unravel_index(flat_index, Y.shape)
    
    drug_id = ligand_ids[drug_row]
    smiles = ligands[drug_id]
    
    protein_name = protein_names[protein_col]
    sequence = proteins[protein_name]
    
    raw_kd_nM = Y[drug_row, protein_col]
    pKd = -np.log10(raw_kd_nM / 1e9)
    
    drug_x, drug_edge_index = smiles_to_graph(smiles)
    protein_indices = sequence_to_indices(sequence)
    
    return drug_x, drug_edge_index, protein_indices, pKd, drug_id, protein_name

In [17]:
flat_idx = train_indices[0]
drug_x, drug_edge_index, protein_indices, pKd, drug_id, protein_name = get_example(flat_idx)

print('Flat index:', flat_idx)
print('Drug ID:', drug_id, '| num atoms:', drug_x.shape[0])
print('Protein name:', protein_name, '| sequence length:', protein_indices.shape[0])
print('True pKd:', pKd)

Flat index: 1483
Drug ID: 11338033 | num atoms: 25
Protein name: FLT1 | sequence length: 1338
True pKd: 5.0


In [18]:
raw_kd = Y[np.unravel_index(flat_idx, Y.shape)]
print('Raw Kd (nM):', raw_kd)
print('Is this a ceiling value?', raw_kd == 10000.0)

Raw Kd (nM): 10000.0
Is this a ceiling value? True


In [19]:
print('Testing get_example on 5 different training indices:\n')

for flat_idx in train_indices[:5]:
    drug_x, drug_edge_index, protein_indices, pKd, drug_id, protein_name = get_example(flat_idx)
    print(f'idx={flat_idx:6d} | drug={drug_id:>10s} (atoms={drug_x.shape[0]:2d}) | '
          f'protein={protein_name:>10s} (len={protein_indices.shape[0]:4d}) | pKd={pKd:.3f}')

Testing get_example on 5 different training indices:

idx=  1483 | drug=  11338033 (atoms=25) | protein=      FLT1 (len=1338) | pKd=5.000
idx= 18097 | drug=    447077 (atoms=29) | protein=      TRKA (len= 790) | pKd=5.000
idx= 11736 | drug=    123631 (atoms=31) | protein=     MERTK (len= 999) | pKd=5.000
idx= 16204 | drug=  51004351 (atoms=43) | protein=      PAK6 (len= 681) | pKd=5.000
idx=  2085 | drug=  10184653 (atoms=34) | protein=PIK3CA(Q546K) (len=1069) | pKd=5.000


In [20]:
sample_pKds = [get_example(idx)[3] for idx in train_indices[:200]]
print('Unique pKd values in first 200 training indices:', sorted(set(sample_pKds)))
print('Fraction at ceiling (pKd=5.0) in this sample:', sum(1 for v in sample_pKds if abs(v - 5.0) < 0.001) / len(sample_pKds))

Unique pKd values in first 200 training indices: [np.float64(5.0), np.float64(5.113509274827518), np.float64(5.1249387366083), np.float64(5.154901959985743), np.float64(5.251811972993799), np.float64(5.376750709602099), np.float64(5.552841968657781), np.float64(5.585026652029182), np.float64(5.638272163982407), np.float64(5.657577319177793), np.float64(5.721246399047171), np.float64(5.7447274948966935), np.float64(5.769551078621726), np.float64(5.795880017344075), np.float64(5.823908740944319), np.float64(5.920818753952375), np.float64(6.086186147616283), np.float64(6.113509274827518), np.float64(6.1938200260161125), np.float64(6.2076083105017466), np.float64(6.2839966563652006), np.float64(6.301029995663981), np.float64(6.356547323513812), np.float64(6.3979400086720375), np.float64(6.508638306165727), np.float64(6.537602002101044), np.float64(6.585026652029182), np.float64(6.619788758288394), np.float64(6.638272163982407), np.float64(6.657577319177793), np.float64(6.6777807052660805),

In [21]:
import random

random.seed(42)
shuffled_train_indices = train_indices.copy()
random.shuffle(shuffled_train_indices)

print('First 5 shuffled indices:', shuffled_train_indices[:5])

First 5 shuffled indices: [15884, 18720, 11860, 7951, 728]


In [22]:
model = DTIModel()  # reuse the class we built earlier
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

In [23]:
model.train()  # tells PyTorch we're in training mode

losses = []

for step, flat_idx in enumerate(shuffled_train_indices[:50]):
    drug_x, drug_edge_index, protein_indices, true_pKd, drug_id, protein_name = get_example(flat_idx)
    
    optimizer.zero_grad()
    
    predicted_pKd = model(drug_x, drug_edge_index, protein_indices)
    true_pKd_tensor = torch.tensor([true_pKd], dtype=torch.float)
    
    loss = loss_fn(predicted_pKd, true_pKd_tensor)
    
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if step % 10 == 0:
        print(f'Step {step:3d} | true_pKd={true_pKd:.3f} | predicted={predicted_pKd.item():.3f} | loss={loss.item():.3f}')

print('\nFirst 5 losses:', losses[:5])
print('Last 5 losses:', losses[-5:])

Step   0 | true_pKd=5.000 | predicted=0.142 | loss=23.601
Step  10 | true_pKd=5.000 | predicted=0.234 | loss=22.710
Step  20 | true_pKd=5.000 | predicted=0.332 | loss=21.792
Step  30 | true_pKd=5.000 | predicted=0.486 | loss=20.376
Step  40 | true_pKd=5.000 | predicted=0.714 | loss=18.370

First 5 losses: [23.600648880004883, 23.477895736694336, 23.406597137451172, 87.4162826538086, 23.18611717224121]
Last 5 losses: [23.240188598632812, 25.890722274780273, 31.8225040435791, 15.689804077148438, 18.915218353271484]


In [24]:
import time

start = time.time()
for flat_idx in shuffled_train_indices[:100]:
    drug_x, drug_edge_index, protein_indices, true_pKd, drug_id, protein_name = get_example(flat_idx)
    optimizer.zero_grad()
    predicted_pKd = model(drug_x, drug_edge_index, protein_indices)
    true_pKd_tensor = torch.tensor([true_pKd], dtype=torch.float)
    loss = loss_fn(predicted_pKd, true_pKd_tensor)
    loss.backward()
    optimizer.step()
elapsed = time.time() - start

print(f'Time for 100 examples: {elapsed:.2f} seconds')
print(f'Time per example: {elapsed/100*1000:.1f} ms')
print(f'Estimated time for full epoch (25,046 examples): {elapsed/100*25046/60:.1f} minutes')

Time for 100 examples: 0.65 seconds
Time per example: 6.5 ms
Estimated time for full epoch (25,046 examples): 2.7 minutes


In [25]:
import time

model = DTIModel()  # fresh model, start clean
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

NUM_EPOCHS = 3

epoch_losses = []

for epoch in range(NUM_EPOCHS):
    random.shuffle(shuffled_train_indices)  # reshuffle every epoch -- important!
    
    start = time.time()
    running_loss = 0.0
    
    for flat_idx in shuffled_train_indices:
        drug_x, drug_edge_index, protein_indices, true_pKd, drug_id, protein_name = get_example(flat_idx)
        
        optimizer.zero_grad()
        predicted_pKd = model(drug_x, drug_edge_index, protein_indices)
        true_pKd_tensor = torch.tensor([true_pKd], dtype=torch.float)
        loss = loss_fn(predicted_pKd, true_pKd_tensor)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(shuffled_train_indices)
    elapsed = time.time() - start
    epoch_losses.append(avg_loss)
    
    print(f'Epoch {epoch+1}/{NUM_EPOCHS} | avg loss = {avg_loss:.4f} | time = {elapsed/60:.1f} min')

print('\nEpoch losses:', epoch_losses)

Epoch 1/3 | avg loss = 0.8129 | time = 2.4 min
Epoch 2/3 | avg loss = 0.6818 | time = 2.4 min
Epoch 3/3 | avg loss = 0.6597 | time = 2.3 min

Epoch losses: [0.8128972757987103, 0.6817818203046766, 0.6596802490795586]


In [26]:
model.eval()  # tells PyTorch we're evaluating, not training (disables certain training-only behaviors)

predicted_pKds = []
true_pKds = []

with torch.no_grad():  # we don't need gradients/backprop during evaluation -- saves time and memory
    for flat_idx in test_fold:
        drug_x, drug_edge_index, protein_indices, true_pKd, drug_id, protein_name = get_example(flat_idx)
        predicted = model(drug_x, drug_edge_index, protein_indices)
        
        predicted_pKds.append(predicted.item())
        true_pKds.append(true_pKd)

print('Number of test predictions:', len(predicted_pKds))
print('Sample predictions vs true:')
for i in range(5):
    print(f'  predicted={predicted_pKds[i]:.3f} | true={true_pKds[i]:.3f}')

Number of test predictions: 5010
Sample predictions vs true:
  predicted=5.123 | true=5.000
  predicted=5.971 | true=5.000
  predicted=5.080 | true=5.000
  predicted=6.532 | true=8.824
  predicted=5.668 | true=5.000


In [27]:
import numpy as np
from scipy.stats import pearsonr

predicted_arr = np.array(predicted_pKds)
true_arr = np.array(true_pKds)

# Overall metrics
r, _ = pearsonr(predicted_arr, true_arr)
mse = np.mean((predicted_arr - true_arr) ** 2)

print(f'Overall Pearson R: {r:.4f}')
print(f'Overall MSE: {mse:.4f}')

# Split: ceiling vs real binders
is_ceiling = np.abs(true_arr - 5.0) < 0.001
print(f'\nNumber of ceiling-value test examples: {is_ceiling.sum()} ({100*is_ceiling.mean():.1f}%)')
print(f'Number of real-binder test examples: {(~is_ceiling).sum()} ({100*(~is_ceiling).mean():.1f}%)')

# Metrics on JUST the real binders (the harder, more interesting subset)
real_binder_mse = np.mean((predicted_arr[~is_ceiling] - true_arr[~is_ceiling]) ** 2)
real_binder_r, _ = pearsonr(predicted_arr[~is_ceiling], true_arr[~is_ceiling])

print(f'\nReal-binder-only Pearson R: {real_binder_r:.4f}')
print(f'Real-binder-only MSE: {real_binder_mse:.4f}')

Overall Pearson R: 0.4706
Overall MSE: 0.6246

Number of ceiling-value test examples: 3468 (69.2%)
Number of real-binder test examples: 1542 (30.8%)

Real-binder-only Pearson R: 0.3025
Real-binder-only MSE: 1.5673


In [28]:
def concordance_index(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)
    
    concordant = 0
    discordant = 0
    
    # compare every pair (i, j) -- this is O(n^2), fine for 5010 but we'll sample for speed
    sample_size = 2000  # full 5010^2 pairs would be slow; sample pairs instead
    rng = np.random.default_rng(42)
    
    for _ in range(sample_size):
        i, j = rng.choice(n, size=2, replace=False)
        if y_true[i] == y_true[j]:
            continue  # tied true values -- skip, no ranking to score
        
        true_order = y_true[i] > y_true[j]
        pred_order = y_pred[i] > y_pred[j]
        
        if true_order == pred_order:
            concordant += 1
        else:
            discordant += 1
    
    return concordant / (concordant + discordant)

ci = concordance_index(true_arr, predicted_arr)
print(f'Concordance Index (sampled estimate): {ci:.4f}')

Concordance Index (sampled estimate): 0.7338


In [29]:
from rdkit.Chem import AllChem
import numpy as np

def get_ecfp(smiles, n_bits=1024, radius=2):
    mol = Chem.MolFromSmiles(smiles)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)

# test on our original molecule
test_fp = get_ecfp(drug_smiles)
print('Fingerprint shape:', test_fp.shape)
print('Number of bits set to 1:', test_fp.sum())
print('First 50 bits:', test_fp[:50])

Fingerprint shape: (1024,)
Number of bits set to 1: 49
First 50 bits: [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0
 0 0 0 0 0 1 0 0 0 0 0 0 0]


[21:14:13] DEPRECATION WARNING: please use MorganGenerator


In [30]:
def get_aa_composition(sequence, vocab=AMINO_ACID_VOCAB):
    counts = np.array([sequence.count(aa) for aa in vocab], dtype=float)
    composition = counts / len(sequence)  # normalize to fractions, so length doesn't bias the result
    return composition

test_comp = get_aa_composition(proteins[protein_names[0]])
print('Composition vector shape:', test_comp.shape)
print('Composition vector:', test_comp)
print('Sum (should be ~1.0):', test_comp.sum())

Composition vector shape: (21,)
Composition vector: [0.08220604 0.01352758 0.04682622 0.04474506 0.03433923 0.06763788
 0.01560874 0.04162331 0.05098855 0.08740895 0.01352758 0.03433923
 0.09781478 0.11654527 0.03329865 0.08324662 0.06451613 0.05306972
 0.00312175 0.         0.01560874]
Sum (should be ~1.0): 1.0


In [31]:
import xgboost as xgb

def build_baseline_dataset(indices):
    X_drug_list = []
    X_protein_list = []
    y_list = []
    
    for flat_idx in indices:
        drug_row, protein_col = np.unravel_index(flat_idx, Y.shape)
        smiles = ligands[ligand_ids[drug_row]]
        sequence = proteins[protein_names[protein_col]]
        
        ecfp = get_ecfp(smiles)
        comp = get_aa_composition(sequence)
        
        raw_kd = Y[drug_row, protein_col]
        pKd = -np.log10(raw_kd / 1e9)
        
        X_drug_list.append(ecfp)
        X_protein_list.append(comp)
        y_list.append(pKd)
    
    X_drug = np.array(X_drug_list)
    X_protein = np.array(X_protein_list)
    X_combined = np.hstack([X_drug, X_protein])  # concatenate, same idea as our drug+protein vectors
    y = np.array(y_list)
    
    return X_combined, y

print('Building training set...')
X_train, y_train = build_baseline_dataset(train_indices)
print('Training set shape:', X_train.shape)

print('Building test set...')
X_test, y_test = build_baseline_dataset(test_fold)
print('Test set shape:', X_test.shape)

Building training set...


[21:15:44] DEPRECATION WARNING: please use MorganGenerator
[21:15:44] DEPRECATION WARNING: please use MorganGenerator
[21:15:44] DEPRECATION WARNING: please use MorganGenerator
[21:15:44] DEPRECATION WARNING: please use MorganGenerator
[21:15:44] DEPRECATION WARNING: please use MorganGenerator
[21:15:44] DEPRECATION WARNING: please use MorganGenerator
[21:15:44] DEPRECATION WARNING: please use MorganGenerator
[21:15:44] DEPRECATION WARNING: please use MorganGenerator
[21:15:44] DEPRECATION WARNING: please use MorganGenerator
[21:15:44] DEPRECATION WARNING: please use MorganGenerator
[21:15:44] DEPRECATION WARNING: please use MorganGenerator
[21:15:44] DEPRECATION WARNING: please use MorganGenerator
[21:15:44] DEPRECATION WARNING: please use MorganGenerator
[21:15:44] DEPRECATION WARNING: please use MorganGenerator
[21:15:44] DEPRECATION WARNING: please use MorganGenerator
[21:15:44] DEPRECATION WARNING: please use MorganGenerator
[21:15:44] DEPRECATION WARNING: please use MorganGenerat

Training set shape: (25046, 1045)
Building test set...


[21:16:17] DEPRECATION WARNING: please use MorganGenerator
[21:16:17] DEPRECATION WARNING: please use MorganGenerator
[21:16:17] DEPRECATION WARNING: please use MorganGenerator
[21:16:17] DEPRECATION WARNING: please use MorganGenerator
[21:16:17] DEPRECATION WARNING: please use MorganGenerator
[21:16:17] DEPRECATION WARNING: please use MorganGenerator
[21:16:17] DEPRECATION WARNING: please use MorganGenerator
[21:16:17] DEPRECATION WARNING: please use MorganGenerator
[21:16:17] DEPRECATION WARNING: please use MorganGenerator
[21:16:17] DEPRECATION WARNING: please use MorganGenerator
[21:16:17] DEPRECATION WARNING: please use MorganGenerator
[21:16:17] DEPRECATION WARNING: please use MorganGenerator
[21:16:17] DEPRECATION WARNING: please use MorganGenerator
[21:16:17] DEPRECATION WARNING: please use MorganGenerator
[21:16:17] DEPRECATION WARNING: please use MorganGenerator
[21:16:17] DEPRECATION WARNING: please use MorganGenerator
[21:16:17] DEPRECATION WARNING: please use MorganGenerat

Test set shape: (5010, 1045)


In [32]:
import warnings
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')  # silences RDKit's internal warnings, including this deprecation notice

In [33]:
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

print('Training XGBoost...')
xgb_model.fit(X_train, y_train)

xgb_predictions = xgb_model.predict(X_test)

print('Training complete.')
print('\nSample predictions vs true:')
for i in range(5):
    print(f'  predicted={xgb_predictions[i]:.3f} | true={y_test[i]:.3f}')

Training XGBoost...
Training complete.

Sample predictions vs true:
  predicted=5.414 | true=5.000
  predicted=5.305 | true=5.000
  predicted=5.156 | true=5.000
  predicted=6.708 | true=8.824
  predicted=5.547 | true=5.000


In [34]:
r_xgb, _ = pearsonr(xgb_predictions, y_test)
mse_xgb = np.mean((xgb_predictions - y_test) ** 2)

is_ceiling_test = np.abs(y_test - 5.0) < 0.001
real_binder_r_xgb, _ = pearsonr(xgb_predictions[~is_ceiling_test], y_test[~is_ceiling_test])
real_binder_mse_xgb = np.mean((xgb_predictions[~is_ceiling_test] - y_test[~is_ceiling_test]) ** 2)

ci_xgb = concordance_index(y_test, xgb_predictions)

print('=== XGBoost Baseline ===')
print(f'Overall Pearson R:        {r_xgb:.4f}')
print(f'Overall MSE:              {mse_xgb:.4f}')
print(f'Real-binder-only R:       {real_binder_r_xgb:.4f}')
print(f'Real-binder-only MSE:     {real_binder_mse_xgb:.4f}')
print(f'Concordance Index:        {ci_xgb:.4f}')

print('\n=== GNN Model (for comparison) ===')
print(f'Overall Pearson R:        {r:.4f}')
print(f'Overall MSE:              {mse:.4f}')
print(f'Real-binder-only R:       {real_binder_r:.4f}')
print(f'Real-binder-only MSE:     {real_binder_mse:.4f}')
print(f'Concordance Index:        {ci:.4f}')

=== XGBoost Baseline ===
Overall Pearson R:        0.7087
Overall MSE:              0.4036
Real-binder-only R:       0.5632
Real-binder-only MSE:     1.0324
Concordance Index:        0.8263

=== GNN Model (for comparison) ===
Overall Pearson R:        0.4706
Overall MSE:              0.6246
Real-binder-only R:       0.3025
Real-binder-only MSE:     1.5673
Concordance Index:        0.7338


In [5]:
class DrugEncoder(nn.Module):
    def __init__(self, atom_feature_dim=14, hidden_dim=64):
        super().__init__()
        self.conv1 = GCNConv(atom_feature_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.attention_scorer = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        h = self.conv3(h, edge_index)
        scores = F.softmax(self.attention_scorer(h), dim=0)
        return (scores * h).sum(dim=0)

class ProteinEncoder(nn.Module):
    def __init__(self, vocab_size=21, embed_dim=8, hidden_dim=64, kernel_size=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv = nn.Conv1d(embed_dim, hidden_dim, kernel_size=kernel_size)

    def forward(self, sequence_indices):
        embedded = self.embedding(sequence_indices)
        x = embedded.unsqueeze(0).transpose(1, 2)
        conv_out = self.conv(x)
        pooled = conv_out.max(dim=2).values
        return pooled.squeeze(0)

class DTIModel(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.drug_encoder = DrugEncoder(hidden_dim=hidden_dim)
        self.protein_encoder = ProteinEncoder(hidden_dim=hidden_dim)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, drug_x, drug_edge_index, protein_sequence_indices):
        drug_vec = self.drug_encoder(drug_x, drug_edge_index)
        protein_vec = self.protein_encoder(protein_sequence_indices)
        combined = torch.cat([drug_vec, protein_vec])
        return self.mlp(combined)

In [6]:
model = DTIModel(hidden_dim=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

shuffled_train_indices = train_indices.copy()
random.seed(42)

NUM_EPOCHS = 8
epoch_losses = []

for epoch in range(NUM_EPOCHS):
    random.shuffle(shuffled_train_indices)
    start = time.time()
    running_loss = 0.0

    for flat_idx in shuffled_train_indices:
        drug_x, drug_edge_index, protein_indices, true_pKd, drug_id, protein_name = get_example(flat_idx)
        optimizer.zero_grad()
        predicted_pKd = model(drug_x, drug_edge_index, protein_indices)
        true_pKd_tensor = torch.tensor([true_pKd], dtype=torch.float)
        loss = loss_fn(predicted_pKd, true_pKd_tensor)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(shuffled_train_indices)
    epoch_losses.append(avg_loss)
    print(f'Epoch {epoch+1}/{NUM_EPOCHS} | avg loss = {avg_loss:.4f} | time = {(time.time()-start)/60:.1f} min')

print('\nTraining complete. Epoch losses:', epoch_losses)

Epoch 1/8 | avg loss = 0.7979 | time = 3.0 min
Epoch 2/8 | avg loss = 0.6707 | time = 3.0 min
Epoch 3/8 | avg loss = 0.6372 | time = 3.0 min
Epoch 4/8 | avg loss = 0.6379 | time = 3.1 min
Epoch 5/8 | avg loss = 0.6201 | time = 2.8 min
Epoch 6/8 | avg loss = 0.6009 | time = 2.7 min
Epoch 7/8 | avg loss = 0.5848 | time = 2.8 min
Epoch 8/8 | avg loss = 0.5683 | time = 2.8 min

Training complete. Epoch losses: [0.7979338381167895, 0.6707226422502147, 0.6372438987564112, 0.6378597432252094, 0.6200564573632648, 0.6009108547808614, 0.5847947070077448, 0.5683135562585011]


In [7]:
def concordance_index(y_true, y_pred, sample_size=2000, seed=42):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)
    rng = np.random.default_rng(seed)
    concordant, discordant = 0, 0
    for _ in range(sample_size):
        i, j = rng.choice(n, size=2, replace=False)
        if y_true[i] == y_true[j]:
            continue
        if (y_true[i] > y_true[j]) == (y_pred[i] > y_pred[j]):
            concordant += 1
        else:
            discordant += 1
    return concordant / (concordant + discordant)

model.eval()
predicted_pKds = []
true_pKds = []

with torch.no_grad():
    for flat_idx in test_fold:
        drug_x, drug_edge_index, protein_indices, true_pKd, drug_id, protein_name = get_example(flat_idx)
        predicted = model(drug_x, drug_edge_index, protein_indices)
        predicted_pKds.append(predicted.item())
        true_pKds.append(true_pKd)

from scipy.stats import pearsonr

predicted_arr = np.array(predicted_pKds)
true_arr = np.array(true_pKds)

r, _ = pearsonr(predicted_arr, true_arr)
mse = np.mean((predicted_arr - true_arr) ** 2)

is_ceiling = np.abs(true_arr - 5.0) < 0.001
real_binder_r, _ = pearsonr(predicted_arr[~is_ceiling], true_arr[~is_ceiling])
real_binder_mse = np.mean((predicted_arr[~is_ceiling] - true_arr[~is_ceiling]) ** 2)

ci = concordance_index(true_arr, predicted_arr)

print(f'Overall Pearson R: {r:.4f}')
print(f'Overall MSE: {mse:.4f}')
print(f'Real-binder-only R: {real_binder_r:.4f}')
print(f'Real-binder-only MSE: {real_binder_mse:.4f}')
print(f'Concordance Index: {ci:.4f}')

Overall Pearson R: 0.5560
Overall MSE: 0.5874
Real-binder-only R: 0.3693
Real-binder-only MSE: 1.1851
Concordance Index: 0.7719


In [8]:
from rdkit.Chem import AllChem
import xgboost as xgb

def get_ecfp(smiles, n_bits=1024, radius=2):
    mol = Chem.MolFromSmiles(smiles)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)

def get_aa_composition(sequence, vocab=AMINO_ACID_VOCAB):
    counts = np.array([sequence.count(aa) for aa in vocab], dtype=float)
    return counts / len(sequence)

def build_baseline_dataset(indices):
    X_list, y_list = [], []
    for flat_idx in indices:
        drug_row, protein_col = np.unravel_index(flat_idx, Y.shape)
        smiles = ligands[ligand_ids[drug_row]]
        sequence = proteins[protein_names[protein_col]]
        ecfp = get_ecfp(smiles)
        comp = get_aa_composition(sequence)
        raw_kd = Y[drug_row, protein_col]
        pKd = -np.log10(raw_kd / 1e9)
        X_list.append(np.concatenate([ecfp, comp]))
        y_list.append(pKd)
    return np.array(X_list), np.array(y_list)

print('Building training set...')
X_train, y_train = build_baseline_dataset(train_indices)
print('Building test set...')
X_test, y_test = build_baseline_dataset(test_fold)
print('Shapes:', X_train.shape, X_test.shape)

xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_predictions = xgb_model.predict(X_test)

r_xgb, _ = pearsonr(xgb_predictions, y_test)
mse_xgb = np.mean((xgb_predictions - y_test) ** 2)
is_ceiling_test = np.abs(y_test - 5.0) < 0.001
real_binder_r_xgb, _ = pearsonr(xgb_predictions[~is_ceiling_test], y_test[~is_ceiling_test])
real_binder_mse_xgb = np.mean((xgb_predictions[~is_ceiling_test] - y_test[~is_ceiling_test]) ** 2)
ci_xgb = concordance_index(y_test, xgb_predictions)

print(f'\nXGBoost - Overall R: {r_xgb:.4f} | Real-binder R: {real_binder_r_xgb:.4f} | CI: {ci_xgb:.4f}')

Building training set...
Building test set...
Shapes: (25046, 1045) (5010, 1045)

XGBoost - Overall R: 0.7087 | Real-binder R: 0.5632 | CI: 0.8263


In [37]:
model.eval()

predicted_pKds_v2 = []
true_pKds_v2 = []

with torch.no_grad():
    for flat_idx in test_fold:
        drug_x, drug_edge_index, protein_indices, true_pKd, drug_id, protein_name = get_example(flat_idx)
        predicted = model(drug_x, drug_edge_index, protein_indices)
        predicted_pKds_v2.append(predicted.item())
        true_pKds_v2.append(true_pKd)

predicted_arr_v2 = np.array(predicted_pKds_v2)
true_arr_v2 = np.array(true_pKds_v2)

r_v2, _ = pearsonr(predicted_arr_v2, true_arr_v2)
mse_v2 = np.mean((predicted_arr_v2 - true_arr_v2) ** 2)

is_ceiling_v2 = np.abs(true_arr_v2 - 5.0) < 0.001
real_binder_r_v2, _ = pearsonr(predicted_arr_v2[~is_ceiling_v2], true_arr_v2[~is_ceiling_v2])
real_binder_mse_v2 = np.mean((predicted_arr_v2[~is_ceiling_v2] - true_arr_v2[~is_ceiling_v2]) ** 2)

ci_v2 = concordance_index(true_arr_v2, predicted_arr_v2)

print('=== GNN v2 (64-dim, 8 epochs) ===')
print(f'Overall Pearson R:    {r_v2:.4f}')
print(f'Overall MSE:          {mse_v2:.4f}')
print(f'Real-binder-only R:   {real_binder_r_v2:.4f}')
print(f'Real-binder-only MSE: {real_binder_mse_v2:.4f}')
print(f'Concordance Index:    {ci_v2:.4f}')

print('\n=== GNN v1 (16-dim, 3 epochs) ===')
print(f'Overall Pearson R:    {r:.4f}  | Real-binder R: {real_binder_r:.4f}  | CI: {ci:.4f}')

print('\n=== XGBoost Baseline ===')
print(f'Overall Pearson R:    {r_xgb:.4f}  | Real-binder R: {real_binder_r_xgb:.4f}  | CI: {ci_xgb:.4f}')

=== GNN v2 (64-dim, 8 epochs) ===
Overall Pearson R:    0.4844
Overall MSE:          0.6145
Real-binder-only R:   0.2953
Real-binder-only MSE: 1.4983
Concordance Index:    0.7595

=== GNN v1 (16-dim, 3 epochs) ===
Overall Pearson R:    0.4706  | Real-binder R: 0.3025  | CI: 0.7338

=== XGBoost Baseline ===
Overall Pearson R:    0.7087  | Real-binder R: 0.5632  | CI: 0.8263


In [38]:
# Well-established cancer-driver kinases (real, FDA-approved-drug targets) -- checking which exist in Davis by name
known_oncology_kinases = [
    'EGFR', 'ABL1', 'BRAF', 'ALK', 'KIT', 'MET', 'RET', 'ROS1', 
    'FLT3', 'JAK2', 'BCR', 'PDGFRA', 'PDGFRB', 'KDR', 'FGFR1', 
    'FGFR2', 'FGFR3', 'SRC', 'AKT1', 'AKT2', 'MTOR', 'CDK4', 'CDK6',
    'PIK3CA', 'BTK', 'JAK1', 'JAK3', 'MAPK1', 'MAPK3', 'AURKA', 'AURKB'
]

found_in_davis = []
for name in known_oncology_kinases:
    matches = [p for p in protein_names if name in p]
    if matches:
        found_in_davis.append((name, matches))

print(f'Found {len(found_in_davis)} out of {len(known_oncology_kinases)} known oncology kinases in Davis:\n')
for name, matches in found_in_davis:
    print(f'{name:10s} -> {matches}')

Found 26 out of 31 known oncology kinases in Davis:

EGFR       -> ['EGFR', 'EGFR(E746A750del)', 'EGFR(G719C)', 'EGFR(G719S)', 'EGFR(L747E749del)', 'EGFR(L747S752del)', 'EGFR(L747T751del)', 'EGFR(L858R)', 'EGFR(L858RT790M)', 'EGFR(L861Q)', 'EGFR(S752I759del)', 'EGFR(T790M)', 'VEGFR2']
ABL1       -> ['ABL1(E255K)', 'ABL1(F317I)', 'ABL1(F317I)p', 'ABL1(F317L)', 'ABL1(F317L)p', 'ABL1(H396P)', 'ABL1(H396P)p', 'ABL1(M351T)', 'ABL1(Q252H)', 'ABL1(Q252H)p', 'ABL1(T315I)', 'ABL1(T315I)p', 'ABL1(Y253F)', 'ABL1', 'ABL1p']
BRAF       -> ['BRAF', 'BRAF(V600E)']
ALK        -> ['ALK']
KIT        -> ['KIT', 'KIT(A829P)', 'KIT(D816H)', 'KIT(D816V)', 'KIT(L576P)', 'KIT(V559D)', 'KIT(V559D-T670I)', 'KIT(V559D-V654A)']
MET        -> ['MET', 'MET(M1250T)', 'MET(Y1235D)']
RET        -> ['RET', 'RET(M918T)', 'RET(V804L)', 'RET(V804M)']
ROS1       -> ['ROS1']
FLT3       -> ['FLT3', 'FLT3(D835H)', 'FLT3(D835Y)', 'FLT3(ITD)', 'FLT3(K663Q)', 'FLT3(N841I)', 'FLT3(R834Q)']
JAK2       -> ['JAK2(JH1domain-catalytic

In [39]:
oncology_panel = sorted(set(
    match for name, matches in found_in_davis for match in matches
))

print(f'Final oncology target panel: {len(oncology_panel)} proteins\n')
for p in oncology_panel:
    print(p)

Final oncology target panel: 83 proteins

ABL1
ABL1(E255K)
ABL1(F317I)
ABL1(F317I)p
ABL1(F317L)
ABL1(F317L)p
ABL1(H396P)
ABL1(H396P)p
ABL1(M351T)
ABL1(Q252H)
ABL1(Q252H)p
ABL1(T315I)
ABL1(T315I)p
ABL1(Y253F)
ABL1p
AKT1
AKT2
ALK
AURKA
AURKB
BRAF
BRAF(V600E)
BTK
CDK4-cyclinD1
CDK4-cyclinD3
EGFR
EGFR(E746A750del)
EGFR(G719C)
EGFR(G719S)
EGFR(L747E749del)
EGFR(L747S752del)
EGFR(L747T751del)
EGFR(L858R)
EGFR(L858RT790M)
EGFR(L861Q)
EGFR(S752I759del)
EGFR(T790M)
FGFR1
FGFR2
FGFR3
FGFR3(G697C)
FLT3
FLT3(D835H)
FLT3(D835Y)
FLT3(ITD)
FLT3(K663Q)
FLT3(N841I)
FLT3(R834Q)
JAK1(JH1domain-catalytic)
JAK1(JH2domain-pseudokinase)
JAK2(JH1domain-catalytic)
JAK3(JH1domain-catalytic)
KIT
KIT(A829P)
KIT(D816H)
KIT(D816V)
KIT(L576P)
KIT(V559D)
KIT(V559D-T670I)
KIT(V559D-V654A)
MET
MET(M1250T)
MET(Y1235D)
MTOR
PDGFRA
PDGFRB
PIK3CA
PIK3CA(C420R)
PIK3CA(E542K)
PIK3CA(E545A)
PIK3CA(E545K)
PIK3CA(H1047L)
PIK3CA(H1047Y)
PIK3CA(I800L)
PIK3CA(M1043I)
PIK3CA(Q546K)
RET
RET(M918T)
RET(V804L)
RET(V804M)
ROS1
SRC
VEGF

In [41]:
import pandas as pd
candidates = {
    'metformin': 'CN(C)C(=N)N=C(N)N',
    'aspirin': 'CC(=O)OC1=CC=CC=C1C(=O)O',
    'simvastatin': r'O=C(O[C@@H]1[C@H]3C(=C/[C@H](C)C1)\C=C/[C@@H]([C@@H]3CC[C@H]2OC(=O)C[C@H](O)C2)C)C(C)(C)CC',
    'loratadine': 'CCOC(=O)N1CCC(=C2c3cc(Cl)ccc3CCc4cccnc24)CC1'
}

results = []

for drug_name, smiles in candidates.items():
    ecfp = get_ecfp(smiles)
    for protein_name in oncology_panel:
        sequence = proteins[protein_name]
        comp = get_aa_composition(sequence)
        features = np.concatenate([ecfp, comp]).reshape(1, -1)
        predicted_pKd = xgb_model.predict(features)[0]
        results.append((drug_name, protein_name, predicted_pKd))

results_df = pd.DataFrame(results, columns=['drug', 'protein', 'predicted_pKd'])
results_df = results_df.sort_values('predicted_pKd', ascending=False)

print('Top 15 predicted drug-target pairs:')
print(results_df.head(15).to_string(index=False))

Top 15 predicted drug-target pairs:
      drug          protein  predicted_pKd
loratadine      FLT3(D835H)       6.391366
loratadine             FLT3       6.391366
loratadine        FLT3(ITD)       6.391366
loratadine      FLT3(R834Q)       6.391366
loratadine      FLT3(K663Q)       6.391366
loratadine      FLT3(N841I)       6.391366
loratadine      FLT3(D835Y)       6.391366
loratadine           PDGFRB       6.386376
loratadine       KIT(D816V)       6.292258
loratadine       KIT(D816H)       6.292258
loratadine       KIT(L576P)       6.292258
loratadine       KIT(V559D)       6.292258
loratadine KIT(V559D-T670I)       6.292258
loratadine KIT(V559D-V654A)       6.292258
loratadine              KIT       6.292258


In [3]:
import mlflow

mlflow.set_experiment("DTI_Davis_Prediction")

with mlflow.start_run(run_name="GNN_v1_16dim_3epoch"):
    mlflow.log_param("hidden_dim", 16)
    mlflow.log_param("num_epochs", 3)
    mlflow.log_param("model_type", "GNN")
    mlflow.log_metric("overall_pearson_r", r)
    mlflow.log_metric("overall_mse", mse)
    mlflow.log_metric("real_binder_r", real_binder_r)
    mlflow.log_metric("real_binder_mse", real_binder_mse)
    mlflow.log_metric("concordance_index", ci)

with mlflow.start_run(run_name="GNN_v2_64dim_8epoch"):
    mlflow.log_param("hidden_dim", 64)
    mlflow.log_param("num_epochs", 8)
    mlflow.log_param("model_type", "GNN")
    mlflow.log_metric("overall_pearson_r", r_v2)
    mlflow.log_metric("overall_mse", mse_v2)
    mlflow.log_metric("real_binder_r", real_binder_r_v2)
    mlflow.log_metric("real_binder_mse", real_binder_mse_v2)
    mlflow.log_metric("concordance_index", ci_v2)

with mlflow.start_run(run_name="XGBoost_ECFP_baseline"):
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("model_type", "XGBoost_ECFP")
    mlflow.log_metric("overall_pearson_r", r_xgb)
    mlflow.log_metric("overall_mse", mse_xgb)
    mlflow.log_metric("real_binder_r", real_binder_r_xgb)
    mlflow.log_metric("real_binder_mse", real_binder_mse_xgb)
    mlflow.log_metric("concordance_index", ci_xgb)

print("All 3 runs logged.")

NameError: name 'r' is not defined

In [2]:
import sys
print(sys.executable)

C:\Users\jimit\OneDrive\Desktop\DTI-project\venv\Scripts\python.exe


In [4]:
import json
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import time
from rdkit import Chem
from rdkit import RDLogger
from torch_geometric.nn import GCNConv

RDLogger.DisableLog('rdApp.*')

DATA_DIR = '../data/GraphDTA/data/davis'

with open(f'{DATA_DIR}/ligands_can.txt') as f:
    ligands = json.load(f)
with open(f'{DATA_DIR}/proteins.txt') as f:
    proteins = json.load(f)
with open(f'{DATA_DIR}/Y', 'rb') as f:
    Y = pickle.load(f, encoding='latin1')
with open(f'{DATA_DIR}/folds/train_fold_setting1.txt') as f:
    train_folds = json.load(f)
with open(f'{DATA_DIR}/folds/test_fold_setting1.txt') as f:
    test_fold = json.load(f)

train_indices = [idx for fold in train_folds for idx in fold]
ligand_ids = list(ligands.keys())
protein_names = list(proteins.keys())

ATOM_VOCAB = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'P', 'I']
AMINO_ACID_VOCAB = sorted(set(letter for seq in proteins.values() for letter in seq))
aa_to_index = {aa: i for i, aa in enumerate(AMINO_ACID_VOCAB)}

def one_hot(value, vocab):
    vec = [0] * (len(vocab) + 1)
    if value in vocab:
        vec[vocab.index(value)] = 1
    else:
        vec[-1] = 1
    return vec

def atom_features(atom):
    features = []
    features += one_hot(atom.GetSymbol(), ATOM_VOCAB)
    features += [atom.GetDegree()]
    features += [atom.GetFormalCharge()]
    features += [int(atom.GetIsAromatic())]
    features += [int(atom.IsInRing())]
    return features

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    all_atom_features = [atom_features(atom) for atom in mol.GetAtoms()]
    edge_sources, edge_targets = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_sources += [i, j]
        edge_targets += [j, i]
    x = torch.tensor(all_atom_features, dtype=torch.float)
    edge_index = torch.tensor([edge_sources, edge_targets], dtype=torch.long)
    return x, edge_index

def sequence_to_indices(sequence):
    return torch.tensor([aa_to_index[letter] for letter in sequence], dtype=torch.long)

def get_example(flat_index):
    drug_row, protein_col = np.unravel_index(flat_index, Y.shape)
    drug_id = ligand_ids[drug_row]
    smiles = ligands[drug_id]
    protein_name = protein_names[protein_col]
    sequence = proteins[protein_name]
    raw_kd_nM = Y[drug_row, protein_col]
    pKd = -np.log10(raw_kd_nM / 1e9)
    drug_x, drug_edge_index = smiles_to_graph(smiles)
    protein_indices = sequence_to_indices(sequence)
    return drug_x, drug_edge_index, protein_indices, pKd, drug_id, protein_name

print('Data loaded. Drugs:', len(ligands), '| Proteins:', len(proteins), '| Train pairs:', len(train_indices), '| Test pairs:', len(test_fold))

Data loaded. Drugs: 68 | Proteins: 442 | Train pairs: 25046 | Test pairs: 5010


In [9]:
import mlflow

mlflow.set_experiment("DTI_Davis_Prediction")

with mlflow.start_run(run_name="GNN_64dim_8epoch_run2"):
    mlflow.log_param("hidden_dim", 64)
    mlflow.log_param("num_epochs", 8)
    mlflow.log_param("model_type", "GNN")
    mlflow.log_metric("overall_pearson_r", r)
    mlflow.log_metric("overall_mse", mse)
    mlflow.log_metric("real_binder_r", real_binder_r)
    mlflow.log_metric("real_binder_mse", real_binder_mse)
    mlflow.log_metric("concordance_index", ci)

with mlflow.start_run(run_name="XGBoost_ECFP_baseline"):
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("model_type", "XGBoost_ECFP")
    mlflow.log_metric("overall_pearson_r", r_xgb)
    mlflow.log_metric("overall_mse", mse_xgb)
    mlflow.log_metric("real_binder_r", real_binder_r_xgb)
    mlflow.log_metric("real_binder_mse", real_binder_mse_xgb)
    mlflow.log_metric("concordance_index", ci_xgb)

print("Both runs logged.")

Both runs logged.
